# UG-Search multi-region: one-click Colab run

Choose a GPU runtime, then **Runtime → Run all**. This notebook clones submodules, installs the harness, prepares both datasets, validates the official baseline, runs Step 0/1 on V* multi-object questions, and writes one comparison table to `results/`. Oracle is V*-only because HR-Bench publishes no boxes.


In [ ]:
import os, pathlib, subprocess
REPO = "https://github.com/yagizdas/projects_main.git"
ROOT = pathlib.Path("/content/projects_main")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--recurse-submodules", REPO, str(ROOT)], check=True)
os.chdir(ROOT / "ideas/ug-search-multiregion")
subprocess.run(["git", "submodule", "update", "--init", "--recursive"], check=True)
subprocess.run(["pip", "install", "-q", "-e", ".[data,gpu]"], check=True)


In [ ]:
import torch
memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0
MODEL = "Qwen/Qwen2.5-VL-7B-Instruct"
if memory_gb < 22:
    print(f"WARNING: only {memory_gb:.1f} GB GPU RAM detected; the 7B default may OOM.")
    print("Running the offered 3B pilot. Set MODEL to the 7B ID and rerun if desired.")
    MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
print("Model:", MODEL)


In [ ]:
# Downloads can take a while: HR-Bench is roughly 10 GB on the Hub.
subprocess.run(["prepare-data", "--output", "data"], check=True)


In [ ]:
# Official baseline first. This is the paper configuration; 86.9 is the paper reference.
subprocess.run(["ug-baseline", "--output", "results/ug-baseline"], check=True)


In [ ]:
common = ["--benchmark", "vstar", "--dataset", "data/vstar/annotations.json",
          "--model", MODEL, "--k", "3"]
subprocess.run(["ug-step0", *common, "--output", "results/step0"], check=True)
subprocess.run(["ug-step1", *common, "--output", "results/step1"], check=True)


In [ ]:
import csv, json, re
from pathlib import Path
rows = []
for stage in ("step0", "step1"):
    for item in json.loads(Path(f"results/{stage}/results.json").read_text()):
        rows.append({"run": stage, "setting": item["setting"], "accuracy": item["accuracy"] * 100})
# lmms-eval result filenames include timestamps; locate the official aggregate defensively.
baseline = None
for path in Path("results/ug-baseline").rglob("*.json"):
    try:
        payload = json.loads(path.read_text())
        value = payload.get("results", {}).get("vstar_bench", {}).get("vstar_overall_acc")
        if value is not None: baseline = float(value) * (100 if float(value) <= 1 else 1)
    except (ValueError, OSError): pass
rows.insert(0, {"run": "official baseline", "setting": "ug_top1", "accuracy": baseline})
rows.insert(1, {"run": "paper Table 1", "setting": "reported", "accuracy": 86.9})
Path("results").mkdir(exist_ok=True)
with Path("results/comparison.csv").open("w", newline="") as f:
    writer=csv.DictWriter(f, fieldnames=["run","setting","accuracy"]); writer.writeheader(); writer.writerows(rows)
lines=["| run | setting | accuracy (%) |","|---|---|---:|"] + [f"| {r['run']} | {r['setting']} | {r['accuracy']:.2f} |" if r['accuracy'] is not None else f"| {r['run']} | {r['setting']} | unavailable |" for r in rows]
Path("results/comparison.md").write_text("\n".join(lines)+"\n")
print("\n".join(lines))
